In [1]:
import re
from collections import Counter
from typing import Dict, List, Tuple
import json
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams

def preprocess_text(text: str) -> str:
    """Clean and preprocess text."""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

def analyze_text_content(content_dict: Dict[str, List[str]]) -> Dict:
    """Analyze text content from URL-chunk dictionary."""
    # Initialize NLTK resources
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('punkt')
        nltk.download('stopwords')
    
    stop_words = set(stopwords.words('english'))
    
    # Analysis results
    results = {
        'total_urls': len(content_dict),
        'word_frequencies': Counter(),
        'bigram_frequencies': Counter(),
        'avg_chunk_length': 0,
        'chunk_length_distribution': [],
        'common_phrases': Counter(),
        'vocabulary_size': set(),
        'url_word_density': {}
    }
    
    total_chunks = 0
    
    for url, chunks in content_dict.items():
        url_text = ' '.join(chunks)
        processed_text = preprocess_text(url_text)
        
        # Word-level analysis
        words = [w for w in word_tokenize(processed_text) if w not in stop_words]
        results['word_frequencies'].update(words)
        results['vocabulary_size'].update(words)
        
        # Bigram analysis
        bigrams = list(ngrams(words, 2))
        bigram_phrases = [' '.join(bg) for bg in bigrams]
        results['bigram_frequencies'].update(bigram_phrases)
        
        # Chunk length distribution
        chunk_lengths = [len(chunk.split()) for chunk in chunks]
        results['chunk_length_distribution'].extend(chunk_lengths)
        total_chunks += len(chunks)
        
        # URL-specific word density
        results['url_word_density'][url] = len(words)
    
    # Calculate averages and clean up results
    results['avg_chunk_length'] = sum(results['chunk_length_distribution']) / total_chunks
    results['vocabulary_size'] = len(results['vocabulary_size'])
    
    # Get top frequencies
    results['top_words'] = dict(results['word_frequencies'].most_common(20))
    results['top_bigrams'] = dict(results['bigram_frequencies'].most_common(20))
    
    return results

def analyze_semantic_overlap(text1: str, text2: str) -> float:
    """Calculate semantic overlap between two texts using basic word overlap."""
    # Preprocess texts
    words1 = set(preprocess_text(text1).split())
    words2 = set(preprocess_text(text2).split())
    
    # Calculate Jaccard similarity
    overlap = len(words1.intersection(words2))
    union = len(words1.union(words2))
    
    return overlap / union if union > 0 else 0

def analyze_chunk_coherence(chunks: List[str]) -> List[float]:
    """Analyze coherence between consecutive chunks."""
    coherence_scores = []
    for i in range(len(chunks) - 1):
        score = analyze_semantic_overlap(chunks[i], chunks[i+1])
        coherence_scores.append(score)
    return coherence_scores

In [4]:
import json
from typing import Dict, List
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def run_text_analysis(file_path: str):
    """Run complete text analysis on the JSON file."""
    
    # Load the JSON data
    with open(file_path, 'r', encoding='utf-8') as f:
        url_chunk_mapping = json.load(f)
    
    # Run the main analysis
    analysis_results = analyze_text_content(url_chunk_mapping)
    
    # Print summary statistics
    print("\n=== Analysis Summary ===")
    print(f"Total URLs analyzed: {analysis_results['total_urls']}")
    print(f"Total unique words: {analysis_results['vocabulary_size']}")
    print(f"Average chunk length: {analysis_results['avg_chunk_length']:.2f} words")
    
    # Print top words and phrases
    print("\n=== Top 10 Most Common Words ===")
    for word, count in list(analysis_results['top_words'].items())[:10]:
        print(f"{word}: {count}")
    
    print("\n=== Top 10 Most Common Bigrams ===")
    for bigram, count in list(analysis_results['top_bigrams'].items())[:10]:
        print(f"{bigram}: {count}")
    
    # Analyze chunk coherence for each URL
    print("\n=== Chunk Coherence Analysis ===")
    coherence_by_url = {}
    for url, chunks in url_chunk_mapping.items():
        coherence_scores = analyze_chunk_coherence(chunks)
        if coherence_scores:
            coherence_by_url[url] = sum(coherence_scores) / len(coherence_scores)
    
    avg_coherence = sum(coherence_by_url.values()) / len(coherence_by_url)
    print(f"Average chunk coherence: {avg_coherence:.3f}")
    
    # Analyze URL word density distribution
    print("\n=== URL Word Density Distribution ===")
    densities = list(analysis_results['url_word_density'].values())
    avg_density = sum(densities) / len(densities)
    print(f"Average words per URL: {avg_density:.2f}")
    
    return analysis_results

if __name__ == "__main__":
    # Replace with your actual file path
    file_path = "url_chunk_mapping_500.json"
    results = run_text_analysis(file_path)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abdal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdal\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.



=== Analysis Summary ===
Total URLs analyzed: 134
Total unique words: 3538
Average chunk length: 63.57 words

=== Top 10 Most Common Words ===
service: 1822
mohap: 986
medical: 720
license: 715
health: 593
application: 563
website: 486
gov: 479
must: 474
smart: 430

=== Top 10 Most Common Bigrams ===
mohap gov: 470
service completion: 253
completion duration: 233
www mohap: 223
website www: 219
duration working: 217
service linked: 201
mohap website: 185
service process: 181
website smart: 172

=== Chunk Coherence Analysis ===
Average chunk coherence: 0.284

=== URL Word Density Distribution ===
Average words per URL: 402.97


In [5]:
file_path = "url_chunk_mapping_1000_v2.0.json"
results = run_text_analysis(file_path)


=== Analysis Summary ===
Total URLs analyzed: 134
Total unique words: 3538
Average chunk length: 127.64 words

=== Top 10 Most Common Words ===
service: 4398
license: 3272
application: 3094
health: 2083
must: 1956
facility: 1870
mohap: 1801
customer: 1692
healthcare: 1619
fee: 1603

=== Top 10 Most Common Bigrams ===
health prevention: 1033
ministry health: 994
employee concerned: 925
facility must: 876
mohap gov: 848
fee payment: 785
customer facility: 777
completion duration: 735
duration working: 705
service completion: 685

=== Chunk Coherence Analysis ===
Average chunk coherence: 0.278

=== URL Word Density Distribution ===
Average words per URL: 970.89
